In [11]:
import pandas as pd
from xgboost import XGBRegressor
import joblib
import matplotlib.pyplot as plt
import glob 
import os
from sklearn.metrics import mean_absolute_error

files = glob.glob("Data/*.csv")
df_list = []

# Goes through all the CSV files in the Data folder
for file in files:
    temp = pd.read_csv(file)
    season = os.path.basename(file).replace(".csv", "")
    temp["Season"] = season
    
    df_list.append(temp)
    

df = pd.concat(df_list, ignore_index=True)

df = df.copy()

tot_players = df[df["Team"] == "TOT"]["Player"].unique()

df = df[(df["Team"] == "TOT") | (~df["Player"].isin(tot_players))]

df = df[df["G"] >= 20]

df["SeasonYear"] = df["Season"].str[:4].astype(int)
df["AgeSquared"] = df["Age"] ** 2

df = df.sort_values(by = ["Player", "SeasonYear"]).reset_index(drop=True)

df["Target_PTS"] = (df.groupby("Player")["PTS"].shift(-1))
df["Target_TRB"] = (df.groupby("Player")["TRB"].shift(-1))
df["Target_AST"] = (df.groupby("Player")["AST"].shift(-1))
df["Target_STL"] = (df.groupby("Player")["STL"].shift(-1))
df["Target_BLK"] = (df.groupby("Player")["BLK"].shift(-1))

df["Prev_PTS"] = df.groupby("Player")["PTS"].shift(1)
df["Prev_TRB"] = df.groupby("Player")["TRB"].shift(1)
df["Prev_AST"] = df.groupby("Player")["AST"].shift(1)
df["Prev_STL"] = df.groupby("Player")["STL"].shift(1)
df["Prev_BLK"] = df.groupby("Player")["BLK"].shift(1)

df["PTS_Change"] = df["PTS"] - df["Prev_PTS"]
df["TRB_Change"] = df["TRB"] - df["Prev_TRB"]
df["AST_Change"] = df["AST"] - df["Prev_AST"]
df["STL_Change"] = df["STL"] - df["Prev_STL"]
df["BLK_Change"] = df["BLK"] - df["Prev_BLK"]

df["PTS_Rolling3"] = (df.groupby("Player")["PTS"].transform(lambda x: x.rolling(3, min_periods=1).mean()))

df = df.dropna(subset=["Target_PTS", "Target_TRB", "Target_AST", "Target_STL", "Target_BLK"])

trend_cols = [
    "Prev_PTS",
    "Prev_TRB",
    "Prev_AST",
    "Prev_STL",
    "Prev_BLK",
    "PTS_Change",
    "TRB_Change",
    "AST_Change",
    "STL_Change",
    "BLK_Change"
]

df[trend_cols] = df[trend_cols].fillna(0)

# Features 
features = [
    "AgeSquared",
    "G",
    "GS",
    "MP",
    "FG",
    "FGA",
    "FG%",
    "3P",
    "3PA",
    "3P%",
    "FT",
    "FTA",
    "FT%",
    "TRB",
    "DRB",
    "ORB",
    "AST",
    "STL",
    "BLK",
    "TOV",
    "PTS",
    "Prev_PTS",
    "Prev_TRB",
    "Prev_AST",
    "Prev_STL",
    "Prev_BLK",
    "PTS_Change",
    "TRB_Change",
    "AST_Change",
    "STL_Change",
    "BLK_Change",
    "PTS_Rolling3"
]

# Train on seasons before 2024
train_df = df[df["SeasonYear"] < 2024]

# Test on 2024 and later
test_df = df[df["SeasonYear"] >= 2024]

# Training features
X_train = train_df[features]

# Testing features
X_test = test_df[features]

# Training targets
y_train_pts = train_df["Target_PTS"]
y_train_trb = train_df["Target_TRB"]
y_train_ast = train_df["Target_AST"]
y_train_stl = train_df["Target_STL"]
y_train_blk = train_df["Target_BLK"]

# Testing targets
y_test_pts = test_df["Target_PTS"]
y_test_trb = test_df["Target_TRB"]
y_test_ast = test_df["Target_AST"]
y_test_stl = test_df["Target_STL"]
y_test_blk = test_df["Target_BLK"]

# Create models
model_pts = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

model_trb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

model_ast = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

model_stl = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

model_blk = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# Train models ONLY on training data
model_pts.fit(X_train, y_train_pts)
model_trb.fit(X_train, y_train_trb)
model_ast.fit(X_train, y_train_ast)
model_stl.fit(X_train, y_train_stl)
model_blk.fit(X_train, y_train_blk)

# Saves models to disk
joblib.dump(model_pts, "Models/pts_model.pkl")
joblib.dump(model_trb, "Models/trb_model.pkl")
joblib.dump(model_ast, "Models/ast_model.pkl")
joblib.dump(model_stl, "Models/stl_model.pkl")
joblib.dump(model_blk, "Models/blk_model.pkl")


preds = model_pts.predict(X_test)

mae = mean_absolute_error(y_test_pts, preds)
print(mae)

def predict_player(row):
    pts = model_pts.predict(row)[0]
    trb = model_trb.predict(row)[0]
    ast = model_ast.predict(row)[0]
    stl = model_stl.predict(row)[0]
    blk = model_blk.predict(row)[0]

    return pts, trb, ast, stl, blk



1.9613173577977325
